In [19]:
import pandas as pd
import warnings
from mlxtend.frequent_patterns import apriori, association_rules

# Mematikan tampilan DeprecationWarning dari pustaka pihak ketiga agar output bersih
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ==========================================
# KONFIGURASI PARAMETER (Bisa Anda Ubah)
# ==========================================
NAMA_FILE_CSV = 'data_transaksi_warkop.csv'  # Ganti dengan nama file CSV Anda
MIN_SUPPORT = 0.40                                   # Batas minimal support
MIN_CONFIDENCE = 0.60                                # Batas minimal confidence

print(f"## MEMULAI PROSES APRIORI UNIVERSAL UNTUK FILE: {NAMA_FILE_CSV}")
print("=" * 80)

# --- 1. MEMBACA FILE CSV ---
try:
    df_raw = pd.read_csv(NAMA_FILE_CSV)
except FileNotFoundError:
    print(f"Error: File '{NAMA_FILE_CSV}' tidak ditemukan. Pastikan posisinya satu folder dengan skrip ini.")
    exit()
print(f"Min Support : {MIN_SUPPORT*100}%")
print(f"Min Confidence : {MIN_CONFIDENCE*100}% \n" )

# Mendeteksi nama kolom secara otomatis
kolom_id = df_raw.columns[0]
kolom_item = df_raw.columns[1]

print(f" Detected Kolom ID  : {kolom_id}")
print(f" Detected Kolom Item: {kolom_item}\n")

# --- 2. TRANSFROMASI DATA MENJADI MATRIKS BINER BOOLEAN (TRUE/FALSE) ---
transactions = df_raw[kolom_item].apply(lambda x: [item.strip() for item in str(x).split(',')])
all_items = sorted(list(set([item for sublist in transactions for item in sublist])))

# Menggunakan tipe data boolean (True/False) untuk menghindari DeprecationWarning dari mlxtend
binary_data = {}
for item in all_items:
    binary_data[item] = df_raw[kolom_item].apply(lambda x: True if item in [i.strip() for i in str(x).split(',')] else False)

df_biner = pd.DataFrame(binary_data)
df_biner.index = df_raw[kolom_id]

print("### [TAHAPAN 1] MATRIKS DATA TRANSAKSI (Tipe: Boolean)")
print("-" * 65)
# Tampilan konversi visual ke 1/0 hanya untuk display di terminal agar mudah dibaca manusia
df_display = df_biner.astype(int).copy()
df_display.loc['Total Kemunculan'] = df_display.sum()
print(df_display)
print("\n" + "="*80 + "\n")

# --- 3. PROSES ALGORITMA APRIORI (GENERASI FREQUENT ITEMSET) ---
# Menghitung kombinasi item yang memenuhi syarat minimum support
frequent_itemsets = apriori(df_biner, min_support=MIN_SUPPORT, use_colnames=True)

total_transaksi = len(df_biner)
min_muncul_kali = int(total_transaksi * MIN_SUPPORT)

frequent_itemsets['Jumlah Transaksi'] = (frequent_itemsets['support'] * total_transaksi).round().astype(int)
frequent_itemsets['% Support'] = (frequent_itemsets['support'] * 100).round(2).astype(str) + "%"
frequent_itemsets['Jumlah Itemset'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("### [TAHAPAN 2] DAFTAR KOMBINASI ITEM YANG LOLOS MIN. SUPPORT")
print(f"Batas Minimal Support: {MIN_SUPPORT*100}% (Minimal muncul {min_muncul_kali} kali dari {total_transaksi} transaksi)")
print("-" * 75)

if frequent_itemsets.empty:
    print("Tidak ada kombinasi item yang memenuhi batas minimum support yang ditentukan.")
else:
    max_item_length = frequent_itemsets['Jumlah Itemset'].max()
    for length in range(1, max_item_length + 1):
        subset = frequent_itemsets[frequent_itemsets['Jumlah Itemset'] == length]
        if not subset.empty:
            print(f"\n-> Kombinasi {length}-Itemset:")
            display_subset = subset.copy()
            display_subset['Daftar Produk'] = display_subset['itemsets'].apply(lambda x: ", ".join(list(x)))
            print(display_subset[['Daftar Produk', 'Jumlah Transaksi', '% Support']].to_string(index=False))

print("\n" + "="*80 + "\n")

# --- 4. PEMBENTUKAN ATURAN ASOSIASI (ASSOCIATION RULES) ---
print("### [TAHAPAN 3] ATURAN ASOSIASI / STRONG RULES YANG TERBENTUK")
print(f"Batas Minimal Confidence: {MIN_CONFIDENCE*100}%")
print("-" * 85)

if not frequent_itemsets.empty:
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)

    if not rules.empty:
        format_rules = []
        for idx, row in rules.iterrows():
            antecedents = ", ".join(list(row['antecedents']))
            consequents = ", ".join(list(row['consequents']))

            rule_text = f"Jika membeli [{antecedents}] -> Maka akan membeli [{consequents}]"
            support_total = f"{row['support']*100:.1f}%"
            confidence_val = f"{row['confidence']*100:.1f}%"
            lift_ratio = f"{row['lift']:.2f}"

            format_rules.append({
                'Aturan Asosiasi (Rules)': rule_text,
                'Support': support_total,
                'Confidence': confidence_val,
                'Lift Ratio': lift_ratio,
                'raw_confidence': row['confidence'],  # Untuk filter kesimpulan
                'raw_lift': row['lift']              # Untuk filter kesimpulan
            })

        df_rules_final = pd.DataFrame(format_rules)
        # Menampilkan tabel utama tanpa kolom pembantu internal
        print(df_rules_final[['Aturan Asosiasi (Rules)', 'Support', 'Confidence', 'Lift Ratio']].to_string(index=False))
        print("\n*Catatan: Lift Ratio > 1 menunjukkan aturan asosiasi yang kuat dan berkolerasi positif.")

        # -----------------------------------------------------------------
        # --- 5. ANALISIS KESIMPULAN OTOMATIS (BAGIAN TAMBAHAN BARU) ---
        # -----------------------------------------------------------------
        print("\n" + "="*80)
        print("### KESIMPULAN STRATEGIS (ATURAN PALING KUAT)")
        print("=" * 80)

        # A. Mencari aturan dengan tingkat kepastian mutlak (Confidence = 100%)
        pasti_rules = df_rules_final[df_rules_final['raw_confidence'] == 1.0]
        print("A. Aturan Mutlak (Confidence 100%):")
        if not pasti_rules.empty:
            for i, r in enumerate(pasti_rules['Aturan Asosiasi (Rules)'], 1):
                print(f"  {i}. {r}")
            print("  [Arti]: Konsumen yang membeli item tersebut dipastikan selalu membeli pasangan produknya.")
        else:
            print("  (Tidak ditemukan aturan dengan tingkat kepastian 100%)")

        # B. Mencari aturan rekomendasi terbaik (Lift Ratio Tertinggi)
        print("\nB. Aturan Rekomendasi Penempatan Barang Terbaik (Berdasarkan Korelasi Nilai Lift):")
        valid_lift_rules = df_rules_final[df_rules_final['raw_lift'] > 1.0].sort_values(by='Lift Ratio', ascending=False)
        if not valid_lift_rules.empty:
            top_rule = valid_lift_rules.iloc[0]
            print(f"  * Rekomendasi Utama: {top_rule['Aturan Asosiasi (Rules)']}")
            print(f"    Sebab memiliki nilai angkat (Lift Ratio) sebesar {top_rule['Lift Ratio']} (Terkuat).")
            print("    Strategi: Kedua item ini sebaiknya diletakkan di rak yang berdekatan atau dipaketkan (bundling).")
        else:
            print("  (Tidak ditemukan item yang saling memengaruhi secara signifikan)")

    else:
        print("Tidak ada Aturan Asosiasi yang berhasil terbentuk memenuhi batas minimum confidence.")
else:
    print("Aturan asosiasi tidak dapat dihitung karena frequent itemset kosong.")

## MEMULAI PROSES APRIORI UNIVERSAL UNTUK FILE: data_transaksi_warkop.csv
Min Support : 40.0%
Min Confidence : 60.0% 

 Detected Kolom ID  : No_Nota
 Detected Kolom Item: Daftar_Belanjaan

### [TAHAPAN 1] MATRIKS DATA TRANSAKSI (Tipe: Boolean)
-----------------------------------------------------------------
                  Gorengan  Kopi  Snack  Susu  Teh
No_Nota                                           
W-01                     1     1      0     0    0
W-02                     0     0      1     0    1
W-03                     1     1      1     0    0
W-04                     1     0      0     1    0
W-05                     0     1      0     1    0
W-06                     1     0      0     0    1
W-07                     1     1      1     0    0
W-08                     0     0      1     1    0
W-09                     1     1      0     0    1
W-10                     0     0      1     0    1
W-11                     1     1      0     0    0
W-12                     1 